In [ ]:
"""utils_common - Shared utilities for DBSpend360 ETL notebooks.

Imported via: %run ./utils_common

Prerequisites:
    - Must be run in a Databricks notebook context where ``spark``
      (SparkSession) is available as a global.  All database-interacting
      functions rely on this runtime-provided global.
    - ``dbutils`` is NOT used by any utility function; only by the
      calling notebooks themselves.

All functions are parameterized with no mutation side effects at import time.
"""

import logging
from datetime import datetime, timedelta, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType


class DataQualityError(Exception):
    """Data fails quality checks: negative costs, mixed currencies, invalid date windows."""
    pass


class SchemaValidationError(Exception):
    """Source DataFrame schema does not match target table requirements."""
    pass

In [ ]:
_MAX_OVERLAP_DAYS = 90


def setup_logger(name, level=logging.INFO):
    """Create a configured logger with standard formatting."""
    logging.basicConfig(level=level)
    return logging.getLogger(name)


def safe_cache(df):
    """Cache a DataFrame, gracefully skipping on serverless compute.

    Serverless Databricks compute does not support PERSIST TABLE.  This
    wrapper lets the same notebook code run on both classic and serverless
    clusters without modification.
    """
    try:
        return df.cache()
    except Exception:
        return df


def safe_unpersist(df):
    """Unpersist a DataFrame, no-op if caching was unavailable."""
    try:
        df.unpersist()
    except Exception:
        pass


def get_overlap_days(raw_value, min_days=2, max_days=_MAX_OVERLAP_DAYS, logger=None):
    """Parse and validate the overlap_days widget value.

    Clamps to *min_days* if below threshold, to *max_days* if above.
    Returns *min_days* on non-numeric input.
    """
    try:
        overlap = int(raw_value or str(min_days))
    except (ValueError, TypeError):
        if logger:
            logger.warning(
                f"overlap_days='{raw_value}' is not a valid integer; "
                f"defaulting to {min_days}."
            )
        return min_days

    if overlap < min_days:
        if logger:
            logger.warning(
                f"overlap_days={overlap} < {min_days}; forcing to {min_days} "
                f"for cost convergence best practice."
            )
        overlap = min_days
    elif overlap > max_days:
        if logger:
            logger.warning(
                f"overlap_days={overlap} > {max_days}; clamping to {max_days}."
            )
        overlap = max_days

    return overlap


def build_table_fqn(catalog, schema, table_name):
    """Build a fully qualified Delta table reference: ``catalog.schema.table_name``.

    Raises ``ValueError`` if any component is empty or non-string.
    """
    for label, value in [("catalog", catalog), ("schema", schema), ("table_name", table_name)]:
        if not isinstance(value, str) or not value.strip():
            raise ValueError(
                f"build_table_fqn: '{label}' must be a non-empty string, got {value!r}"
            )
    return f"{catalog}.{schema}.{table_name}"

def add_workspace_covered(df, covered_table, workspace_id_col="workspace_id", logger=None):
    """Tag each row with workspace_covered via LEFT JOIN to the covered map.

    When the covered table is empty (e.g. AWS/GCP deployments that skip ARM
    discovery), every row is tagged ``True`` so coverage labeling is a no-op.
 
    On Azure (``subscription_id`` widget set), an empty covered table means
    discovery has not run yet — emit a loud warning so rows are not silently
    mislabeled as covered. The DAB orders ``covered_workspaces`` before all
    DBU rollups so this should not happen in normal operation.
    """
    try:
        covered_count = spark.table(covered_table).limit(1).count()
    except Exception:
        covered_count = 0
    if covered_count == 0:
        try:
            subscription_id = dbutils.widgets.get("subscription_id").strip()
        except Exception:
            subscription_id = ""
        if subscription_id:
            msg = (
                "dbspend360_covered_workspaces is empty on an Azure deployment "
                "(subscription_id widget is set). All rows will be tagged "
                "workspace_covered=true until discovery runs — cross-subscription "
                "workspaces will be mislabeled. Ensure covered_workspaces runs "
                "before DBU rollups."
            )
            if logger is not None:
                logger.warning(msg)
            else:
                print(f"WARNING: {msg}")
        return df.withColumn("workspace_covered", F.lit(True))
    covered_df = (
        spark.table(covered_table)
        .select(F.col("workspace_id").alias("_cw_workspace_id"))
        .distinct()
    )
    return (
        df.join(
            covered_df,
            df[workspace_id_col] == covered_df["_cw_workspace_id"],
            "left",
        )
        .withColumn(
            "workspace_covered",
            F.when(F.col("_cw_workspace_id").isNotNull(), F.lit(True)).otherwise(
                F.lit(False)
            ),
        )
        .drop("_cw_workspace_id")
    )


In [ ]:
def _safe_append(df, table):
    """Append a DataFrame to an existing Delta table with column matching by name.

    Uses the DataFrameWriterV2 API (writeTo) which resolves columns by name,
    not by ordinal position, preventing silent data corruption when the table
    schema is reordered or extended.
    """
    df.writeTo(table).append()


def get_date_window(audit_table, table_name, overlap_days):
    """Determine the incremental date window from the audit log.

    On first run (no prior SUCCESS entries), defaults to 365 days back.
    On subsequent runs, uses the last successful end_date minus
    overlap_days for idempotent re-MERGE coverage.

    Returns:
        Tuple[date, date]: (start_dt, end_dt)
    """
    wm = (
        spark.table(audit_table)
             .filter(f"table_name = '{table_name}' AND status = 'SUCCESS'")
    )

    if wm.limit(1).count() == 0:
        last_end_date = (
            datetime.now(timezone.utc).date()
            - timedelta(days=365 - overlap_days)
        )
    else:
        last_end_date = wm.agg(F.max("end_date")).collect()[0][0]

    start_dt = last_end_date - timedelta(days=overlap_days - 1)
    end_dt = datetime.now(timezone.utc).date()
    return start_dt, end_dt


def validate_date_window(start_dt, end_dt, overlap_days=None):
    """Validate that start_dt <= end_dt.

    Returns:
        Tuple[bool, str]: (is_valid, error_message)
    """
    if start_dt > end_dt:
        msg = f"Invalid date window: start_dt={start_dt} > end_dt={end_dt}."
        if overlap_days is not None:
            msg += f" Check audit table and overlap_days={overlap_days}."
        return False, msg
    return True, ""


def log_audit_run(audit_table, table_name, start_dt, end_dt, status, row_count, message=""):
    """Append a run record to the audit log table.

    Uses explicit column mapping (not positional ``insertInto``) for schema safety.
    """
    run_log_df = spark.createDataFrame([
        Row(
            table_name=table_name,
            start_date=start_dt,
            end_date=end_dt,
            status=status,
            row_count=int(row_count),
            message=str(message or ""),
            created_at=datetime.now(timezone.utc),
        )
    ])
    _safe_append(run_log_df, audit_table)

In [ ]:
def ensure_cost_columns(target_table, columns=None, logger=None):
    """Add missing cost segmentation columns (DOUBLE) to a Delta table.

    Args:
        target_table: fully qualified table name
        columns: column names to ensure exist
                 (default: compute_cost, storage_cost, network_cost, other_cost)
        logger: optional logger for info messages
    """
    if columns is None:
        columns = ["compute_cost", "storage_cost", "network_cost", "other_cost"]
    existing = {c.name for c in spark.table(target_table).schema}
    missing = [c for c in columns if c not in existing]
    if missing:
        cols_sql = ", ".join(f"{c} DOUBLE" for c in missing)
        spark.sql(f"ALTER TABLE {target_table} ADD COLUMNS ({cols_sql})")
        if logger:
            logger.info(f"Added columns to {target_table}: {missing}")


def ensure_boolean_columns(target_table, columns, logger=None):
    """Add missing BOOLEAN columns to a Delta table (e.g. workspace_covered).

    Kept separate from ensure_cost_columns so coverage flags are never
    accidentally created as DOUBLE.
    """
    existing = {c.name for c in spark.table(target_table).schema}
    missing = [c for c in columns if c not in existing]
    if missing:
        cols_sql = ", ".join(f"{c} BOOLEAN" for c in missing)
        spark.sql(f"ALTER TABLE {target_table} ADD COLUMNS ({cols_sql})")
        if logger:
            logger.info(f"Added boolean columns to {target_table}: {missing}")


In [ ]:
def aggregate_costs_by_category(classified_df, cost_col="cost"):
    """Pivot a classified DataFrame into per-category cost columns.

    Expects input DataFrame to have columns:
        cluster_id, currency, cost_incurred_date, category, {cost_col}

    The 'category' column must contain values from: compute, storage, network, other.
    Invariant enforced: cloud_cost = compute_cost + storage_cost + network_cost + other_cost
    """
    return (
        classified_df
        .groupBy("cluster_id", "currency", "cost_incurred_date")
        .agg(
            F.sum(F.when(F.col("category") == "compute", F.col(cost_col)).otherwise(0)).alias("compute_cost"),
            F.sum(F.when(F.col("category") == "storage", F.col(cost_col)).otherwise(0)).alias("storage_cost"),
            F.sum(F.when(F.col("category") == "network", F.col(cost_col)).otherwise(0)).alias("network_cost"),
            F.sum(F.when(F.col("category") == "other", F.col(cost_col)).otherwise(0)).alias("other_cost"),
        )
        .withColumn(
            "cloud_cost",
            F.col("compute_cost") + F.col("storage_cost") + F.col("network_cost") + F.col("other_cost"),
        )
        .withColumn("created_at", F.current_timestamp())
        .withColumn("updated_at", F.current_timestamp())
    )


def compute_quality_metrics(agg_df, row_count, overlap_days, logger=None):
    """Compute classification coverage metrics and return a summary string.

    Measures what fraction of total cost is classified into compute/storage/network
    vs. falling into the 'other' bucket.
    """
    metrics = agg_df.agg(
        F.sum("cloud_cost").alias("total_cost"),
        F.sum("compute_cost").alias("classified_compute"),
        F.sum("storage_cost").alias("classified_storage"),
        F.sum("network_cost").alias("classified_network"),
        F.sum("other_cost").alias("unclassified_cost"),
    ).collect()[0]

    total = float(metrics.total_cost or 0)
    unclassified = float(metrics.unclassified_cost or 0)
    classified = total - unclassified
    coverage_pct = (classified / total * 100) if total > 0 else 100.0

    msg = (
        f"overlap_days={overlap_days}, rows={row_count}, "
        f"classification_coverage={coverage_pct:.1f}%, "
        f"classified_cost={classified:.2f}, "
        f"unclassified_cost={unclassified:.2f}, "
        f"total_cost={total:.2f}"
    )
    if logger:
        logger.info(f"Data quality: {msg}")

    _OTHER_COST_WARN_PCT = 30.0
    other_pct = 100.0 - coverage_pct
    if other_pct > _OTHER_COST_WARN_PCT and logger:
        logger.warning(
            f"Unclassified cost ({other_pct:.1f}%) exceeds {_OTHER_COST_WARN_PCT}% threshold. "
            f"Review classification rules or add new service mappings."
        )

    return msg

In [ ]:
def merge_cloud_cost_explorer(target_table, source_df):
    """MERGE incremental cloud cost data into the cloud cost explorer table.

    Precondition: target_table must already exist (created via DDL in jobs/ddls/).
    Matches on (cluster_id, currency, cost_incurred_date).
    Updates cost columns on match; inserts new rows otherwise.
    Uses DeltaTable API for DataFrame-native merge without temp views.
    """
    target = DeltaTable.forName(spark, target_table)
    (target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.cluster_id = s.cluster_id AND t.currency = s.currency "
            "AND t.cost_incurred_date = s.cost_incurred_date",
        )
        .whenMatchedUpdate(set={
            "cloud_cost": "s.cloud_cost",
            "compute_cost": "s.compute_cost",
            "storage_cost": "s.storage_cost",
            "network_cost": "s.network_cost",
            "other_cost": "s.other_cost",
            "updated_at": "current_timestamp()",
        })
        .whenNotMatchedInsert(values={
            "cluster_id": "s.cluster_id",
            "cloud_cost": "s.cloud_cost",
            "compute_cost": "s.compute_cost",
            "storage_cost": "s.storage_cost",
            "network_cost": "s.network_cost",
            "other_cost": "s.other_cost",
            "currency": "s.currency",
            "cost_incurred_date": "s.cost_incurred_date",
            "created_at": "current_timestamp()",
            "updated_at": "current_timestamp()",
        })
        .execute()
    )


def merge_pool_cloud_cost_explorer(target_table, source_df):
    """MERGE incremental per-pool cloud cost into the pool cloud explorer table.

    Precondition: target_table must already exist (created via DDL in
    jobs/ddls/dbspend360_pool_cloud_cost_explorer).
    Matches on (instance_pool_id, currency, cost_incurred_date).
    Updates the cost columns on match; inserts new rows otherwise. The
    reserved idle_cloud_cost / active_cloud_cost columns are intentionally
    left untouched (NULL until the instance_events split lands - plan §4.5).
    Uses DeltaTable API for DataFrame-native merge without temp views.
    """
    target = DeltaTable.forName(spark, target_table)
    (target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.instance_pool_id = s.instance_pool_id AND t.currency = s.currency "
            "AND t.cost_incurred_date = s.cost_incurred_date",
        )
        .whenMatchedUpdate(set={
            "cloud_cost": "s.cloud_cost",
            "compute_cost": "s.compute_cost",
            "storage_cost": "s.storage_cost",
            "network_cost": "s.network_cost",
            "other_cost": "s.other_cost",
            "updated_at": "current_timestamp()",
        })
        .whenNotMatchedInsert(values={
            "instance_pool_id": "s.instance_pool_id",
            "cloud_cost": "s.cloud_cost",
            "compute_cost": "s.compute_cost",
            "storage_cost": "s.storage_cost",
            "network_cost": "s.network_cost",
            "other_cost": "s.other_cost",
            "currency": "s.currency",
            "cost_incurred_date": "s.cost_incurred_date",
            "created_at": "current_timestamp()",
            "updated_at": "current_timestamp()",
        })
        .execute()
    )


def merge_other_cost_breakdown(breakdown_table, source_df):
    """MERGE other cost breakdown data into the breakdown table.

    Precondition: breakdown_table must already exist (created via DDL in jobs/ddls/).
    Matches on (cost_incurred_date, cluster_id, source_system, service_name, currency).
    Uses DeltaTable API for DataFrame-native merge without temp views.
    """
    target = DeltaTable.forName(spark, breakdown_table)
    (target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.cost_incurred_date = s.cost_incurred_date AND t.cluster_id = s.cluster_id "
            "AND t.source_system = s.source_system AND t.service_name = s.service_name "
            "AND t.currency = s.currency",
        )
        .whenMatchedUpdate(set={
            "cost": "s.cost",
            "updated_at": "current_timestamp()",
        })
        .whenNotMatchedInsert(values={
            "cost_incurred_date": "s.cost_incurred_date",
            "cluster_id": "s.cluster_id",
            "source_system": "s.source_system",
            "service_name": "s.service_name",
            "cost": "s.cost",
            "currency": "s.currency",
            "created_at": "current_timestamp()",
            "updated_at": "current_timestamp()",
        })
        .execute()
    )


def validate_post_merge(target_table, date_col, start_dt, end_dt, expected_count, logger=None):
    """Verify that the target table has at least expected_count rows in the date window.

    Returns (actual_count, warning_message). Empty warning means validation passed.
    """
    actual_count = (
        spark.table(target_table)
        .filter((F.col(date_col) >= F.lit(start_dt)) & (F.col(date_col) <= F.lit(end_dt)))
        .count()
    )
    if actual_count < expected_count:
        msg = (
            f"Post-merge: {target_table} has {actual_count} rows "
            f"for {start_dt} → {end_dt}, expected >= {expected_count}"
        )
        if logger:
            logger.warning(msg)
        return actual_count, msg
    if logger:
        logger.info(f"Post-merge OK: {actual_count} rows in {target_table} for {start_dt} → {end_dt}")
    return actual_count, ""


def get_merge_metrics(table_name, logger=None):
    """Extract row-level metrics from the last MERGE operation via Delta history.

    Returns:
        dict with num_inserted, num_updated, num_deleted, num_source_rows
        (empty dict if metrics unavailable).
    """
    try:
        history = spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").collect()
        if not history:
            return {}
        metrics = history[0].operationMetrics or {}
        result = {
            "num_inserted": int(metrics.get("numTargetRowsInserted", 0)),
            "num_updated": int(metrics.get("numTargetRowsUpdated", 0)),
            "num_deleted": int(metrics.get("numTargetRowsDeleted", 0)),
            "num_source_rows": int(metrics.get("numSourceRows", 0)),
        }
        if logger:
            logger.info(
                f"MERGE metrics [{table_name}]: "
                f"inserted={result['num_inserted']}, "
                f"updated={result['num_updated']}, "
                f"deleted={result['num_deleted']}, "
                f"source_rows={result['num_source_rows']}"
            )
        return result
    except Exception as e:
        if logger:
            logger.warning(f"Could not retrieve MERGE metrics for {table_name}: {e}")
        return {}

In [ ]:
_ERROR_LOG_SCHEMA = StructType([
    StructField("source_system", StringType()),
    StructField("error_type", StringType()),
    StructField("cluster_id", StringType()),
    StructField("job_id", StringType()),
    StructField("run_id", StringType()),
    StructField("usage_date", DateType()),
    StructField("currency", StringType()),
    StructField("error_detail", StringType()),
    StructField("raw_record", StringType()),
    StructField("created_at", TimestampType()),
])


def write_error_log_entries(error_details, source_system, error_type, error_log_table):
    """Write structured error entries to the error_log table.

    Uses explicit column mapping for schema safety.

    Args:
        error_details: list of human-readable error description strings
        source_system: origin identifier, e.g. "AWS", "AZURE", "GCP"
        error_type: classification, e.g. "UNCLASSIFIED_COST"
        error_log_table: fully qualified error log table name
    """
    if not error_details:
        return
    error_records = [
        Row(
            source_system=source_system,
            error_type=error_type,
            cluster_id=None,
            job_id=None,
            run_id=None,
            usage_date=None,
            currency=None,
            error_detail=detail,
            raw_record=None,
            created_at=datetime.now(timezone.utc),
        )
        for detail in error_details
    ]
    _safe_append(
        spark.createDataFrame(error_records, schema=_ERROR_LOG_SCHEMA),
        error_log_table,
    )


def write_other_cost_breakdown(classified_df, service_col, source_system, breakdown_table, logger=None):
    """Write per-service detail for 'other' category costs to the breakdown table.

    Aggregates unclassified costs by (date, cluster, service) and MERGEs
    into the breakdown table. Idempotent: reruns update existing rows.

    Args:
        classified_df: DataFrame with 'category' column already applied
        service_col: column name for the service/meter identifier
        source_system: "AWS", "AZURE", or "GCP"
        breakdown_table: fully qualified breakdown table name
        logger: optional logger instance

    Returns:
        int: number of breakdown rows written (0 if none)
    """
    other_df = (
        classified_df
        .filter(F.col("category") == "other")
        .groupBy("cluster_id", service_col, "currency", "cost_incurred_date")
        .agg(F.sum("cost").alias("cost"))
        .withColumnRenamed(service_col, "service_name")
        .withColumn("source_system", F.lit(source_system))
        .withColumn("created_at", F.current_timestamp())
        .withColumn("updated_at", F.current_timestamp())
    )
    other_df = safe_cache(other_df)

    try:
        if other_df.limit(1).count() == 0:
            if logger:
                logger.info("No 'other' category costs to write to breakdown table.")
            return 0

        merge_other_cost_breakdown(breakdown_table, other_df)

        count = other_df.count()
        if logger:
            logger.info(f"Wrote {count} other cost breakdown rows to {breakdown_table}")
        return count
    finally:
        safe_unpersist(other_df)


def filter_valid_cost_rows(df, cluster_col="cluster_id", date_col="cost_incurred_date"):
    """Filter out rows with null/empty cluster ID or null cost date."""
    return (
        df
        .filter((F.col(cluster_col).isNotNull()) & (F.col(cluster_col) != ""))
        .filter(F.col(date_col).isNotNull())
    )

In [ ]:
# ---------------------------------------------------------------------------
# Data Quality Guards & Schema Safety
# ---------------------------------------------------------------------------

def validate_source_schema(df, required_columns, table_context="", logger=None):
    """Validate that a source DataFrame has required columns (and optionally types) before MERGE.

    Args:
        df: source DataFrame to validate
        required_columns: dict {column_name: expected_spark_type_string or None}.
                          Use None to skip type checking for that column.
        table_context: target table name for error messages
        logger: optional logger

    Raises:
        SchemaValidationError: if required columns are missing or have wrong types
    """
    df_schema = {f.name: f.dataType.simpleString() for f in df.schema.fields}
    missing = [c for c in required_columns if c not in df_schema]
    if missing:
        raise SchemaValidationError(
            f"Source DataFrame for {table_context} is missing required columns: {missing}. "
            f"Available: {sorted(df_schema.keys())}"
        )

    type_mismatches = []
    for col_name, expected_type in required_columns.items():
        if expected_type is None:
            continue
        actual_type = df_schema[col_name]
        if actual_type != expected_type:
            type_mismatches.append(f"{col_name}: expected {expected_type}, got {actual_type}")
    if type_mismatches:
        raise SchemaValidationError(
            f"Schema type mismatches for {table_context}: {'; '.join(type_mismatches)}"
        )

    if logger:
        logger.info(f"Schema validation passed for {table_context}: {len(required_columns)} columns verified")


def validate_no_negative_costs(df, cost_columns, table_context="", logger=None):
    """Check for negative values in cost columns. Logs warnings; does not fail.

    Cloud providers may issue credits as negative line items, so this is
    advisory rather than a hard gate.

    Returns:
        list[str]: warning messages for columns with negative values (empty = clean)
    """
    warnings = []
    for col_name in cost_columns:
        if col_name not in df.columns:
            continue
        neg_count = df.filter(F.col(col_name) < 0).count()
        if neg_count > 0:
            msg = f"{table_context}: {neg_count} rows with negative {col_name}"
            warnings.append(msg)
            if logger:
                logger.warning(msg)
    return warnings


def validate_currency_consistency(df, currency_col="currency", table_context="", logger=None):
    """Verify all rows use a single currency before aggregation.

    Mixed currencies in a SUM would produce meaningless totals.

    Raises:
        DataQualityError: if multiple distinct currencies are found
    """
    currencies = [row[0] for row in df.select(currency_col).distinct().collect()]
    if len(currencies) > 1:
        raise DataQualityError(
            f"{table_context}: Mixed currencies detected: {currencies}. "
            f"Cannot safely aggregate costs across different currencies."
        )
    if logger and currencies:
        logger.info(f"{table_context}: Currency consistency OK ({currencies[0]})")


# Default unpriced-DBU fail threshold (percent of usage quantity). Small but
# non-zero so a whole SKU losing its list price trips the gate while a tiny
# price-window seam does not. Operator-overridable per notebook via the
# ``price_unpriced_fail_pct`` widget (see ``get_price_fail_threshold``).
_PRICE_UNPRICED_FAIL_PCT_DEFAULT = 0.5


def validate_price_coverage(
    joined_df,
    price_col,
    quantity_col,
    sku_col,
    table_context="",
    logger=None,
    fail_threshold_pct=_PRICE_UNPRICED_FAIL_PCT_DEFAULT,
    max_skus_logged=25,
):
    """Detect usage rows that found no matching list_prices row (silent DBU undercount guard).

    DBU dollars are built as ``SUM(usage_quantity * list_prices.pricing.default)``
    over a LEFT join to ``system.billing.list_prices``. When a
    ``(sku_name, price-window)`` has no match, ``pricing.default`` is NULL,
    ``usage_quantity * NULL`` is NULL, and ``SUM`` silently skips it -- the
    resulting ``databricks_cost`` is understated with no error raised and no
    negative value for ``validate_no_negative_costs`` to catch (an all-unpriced
    group even yields ``databricks_cost = NULL``). This guard measures the
    unpriced slice on the SAME joined frame that feeds the aggregation, so what
    it reports is exactly the usage that ``SUM`` would drop.

    Behaviour (see ``get_price_fail_threshold`` for the operator-facing knob):
        * Always returns ``(metrics, summary)`` and logs -- INFO when coverage is
          complete, WARNING when any unpriced usage exists (naming the SKUs).
        * Raises ``DataQualityError`` when the unpriced share of usage quantity
          exceeds ``fail_threshold_pct``. Pass ``fail_threshold_pct=None`` to
          warn only and never fail.

    The unpriced share is measured in usage *quantity* (DBUs), not dollars: the
    dollar value of an unpriced row is by definition unknowable (its price is
    the very thing that is missing), so quantity is the only available proxy.

    Cost note (deliberate): this adds one aggregation pass over the pre-agg
    joined frame purely for the coverage signal -- correctness over speed,
    mirroring the pipeline collector's 1:1 price-join assertion.

    Args:
        joined_df: the ``usage`` LEFT-join ``list_prices`` frame, BEFORE aggregation.
        price_col: Column for the joined price, e.g. ``F.col("list_prices.pricing")["default"]``.
        quantity_col: Column for the usage quantity, e.g. ``F.col("usage.usage_quantity")``.
        sku_col: Column for the usage SKU name, e.g. ``F.col("usage.sku_name")``.
        table_context: target table name, used in log / exception messages.
        logger: optional logger.
        fail_threshold_pct: unpriced-quantity percentage above which to raise
            (``None`` disables the hard fail).
        max_skus_logged: cap on the number of distinct unpriced SKU names listed.

    Returns:
        Tuple[dict, str]: ``(metrics, summary)`` where ``metrics`` has keys
        ``unpriced_rows``, ``unpriced_quantity``, ``total_quantity``,
        ``unpriced_pct`` and ``unpriced_skus`` (list[str]).

    Raises:
        DataQualityError: if the unpriced quantity share exceeds ``fail_threshold_pct``.
    """
    # A usage row only contributes to the SUM when it carries non-zero quantity;
    # a zero-quantity unpriced row drops nothing, so exclude it from the signal.
    # abs() guards the denominator against sign cancellation if a workspace ever
    # emits negative (correction) quantities.
    unpriced = price_col.isNull() & quantity_col.isNotNull() & (quantity_col != 0)

    row = joined_df.agg(
        F.coalesce(F.sum(F.abs(quantity_col)), F.lit(0.0)).alias("total_quantity"),
        F.coalesce(
            F.sum(F.when(unpriced, F.abs(quantity_col))), F.lit(0.0)
        ).alias("unpriced_quantity"),
        F.count(F.when(unpriced, F.lit(1))).alias("unpriced_rows"),
        F.array_sort(F.collect_set(F.when(unpriced, sku_col))).alias("unpriced_skus"),
    ).collect()[0]

    total_qty = float(row["total_quantity"] or 0.0)
    unpriced_qty = float(row["unpriced_quantity"] or 0.0)
    unpriced_rows = int(row["unpriced_rows"] or 0)
    unpriced_skus = [s for s in (row["unpriced_skus"] or []) if s is not None]
    unpriced_pct = (unpriced_qty / total_qty * 100.0) if total_qty > 0 else 0.0

    metrics = {
        "unpriced_rows": unpriced_rows,
        "unpriced_quantity": unpriced_qty,
        "total_quantity": total_qty,
        "unpriced_pct": unpriced_pct,
        "unpriced_skus": unpriced_skus,
    }

    if unpriced_rows == 0:
        summary = "price_coverage=100.00% (0 unpriced usage rows)"
        if logger:
            logger.info(f"{table_context}: {summary}")
        return metrics, summary

    shown = unpriced_skus[:max_skus_logged]
    sku_str = ", ".join(shown)
    if len(unpriced_skus) > max_skus_logged:
        sku_str += f", ... (+{len(unpriced_skus) - max_skus_logged} more)"
    summary = (
        f"price_coverage={100.0 - unpriced_pct:.2f}%, "
        f"unpriced_rows={unpriced_rows}, "
        f"unpriced_quantity={unpriced_qty:.4f}/{total_qty:.4f} DBU "
        f"({unpriced_pct:.2f}%), unpriced_skus=[{sku_str}]"
    )

    if fail_threshold_pct is not None and unpriced_pct > fail_threshold_pct:
        raise DataQualityError(
            f"{table_context}: unpriced DBU usage {unpriced_pct:.2f}% exceeds the "
            f"{fail_threshold_pct:.2f}% threshold -- databricks_cost would be "
            f"silently understated. {summary}. Check system.billing.list_prices "
            f"coverage for those SKUs (newly introduced SKU or a price-window gap)."
        )

    if logger:
        logger.warning(f"{table_context}: {summary}")
    return metrics, summary


def get_price_fail_threshold(raw_value, default=_PRICE_UNPRICED_FAIL_PCT_DEFAULT, logger=None):
    """Parse the ``price_unpriced_fail_pct`` widget for ``validate_price_coverage``.

    Returns a float percentage, or ``None`` to disable the hard fail (warn-only):
        * blank / missing  -> *default*
        * a negative number -> ``None`` (warn only, never fail)
        * non-numeric       -> *default* (with a warning)
    """
    if raw_value is None or str(raw_value).strip() == "":
        return default
    try:
        val = float(raw_value)
    except (ValueError, TypeError):
        if logger:
            logger.warning(
                f"price_unpriced_fail_pct='{raw_value}' is not a number; "
                f"defaulting to {default}."
            )
        return default
    return None if val < 0 else val